### Importação de Pacotes

Para Julia, utilizaremos `Plots` para gráficos e `JuMP` com os solvers `HiGHS` (linear) e `Ipopt` (não-linear).

In [ ]:
using Plots
using JuMP
using HiGHS
using Ipopt

# Exercício: Sin(x)

Plot the graph of the function sin(x) over the interval $[-\pi/4, 3\pi/4]$

In [ ]:
x = range(-pi/4, 3*pi/4, length=100)
y = sin.(x)

plot(x, y, title="Gráfico de sin(x)", xlabel="x", ylabel="sin(x)", lw=2)

Plot the graph of the function $x\cdot sin(x)$ over the interval $[-10\pi, 10\pi]$

In [ ]:
x2 = range(-10*pi, 10*pi, length=500)
y2 = x2 .* sin.(x2)

plot(x2, y2, title="Gráfico de x * sin(x)", xlabel="x", ylabel="f(x)", lw=1.5)

# Problema do Cilindro

Solve the Cylinder Problem considering the following data:

* N: 10
* $c_1$: 2
* $c_2$: 0.5

In [ ]:
model_cilindro = Model(Ipopt.Optimizer)
set_silent(model_cilindro)

N = 10
c1 = 2
c2 = 0.5

@variable(model_cilindro, r >= 0, start = 1.0)
@variable(model_cilindro, h >= 0, start = 1.0)

@NLobjective(model_cilindro, Min, c1 * (2 * pi * r^2) + c2 * (2 * pi * r * h))
@NLconstraint(model_cilindro, pi * r^2 * h == N)

optimize!(model_cilindro)

println("Raio ideal (r): ", round(value(r), digits=4))
println("Altura ideal (h): ", round(value(h), digits=4))
println("Custo total mínimo: ", round(objective_value(model_cilindro), digits=4))

# Problema do Toldo (Awning Problem)

Solve the Awning Problem considering the following data:

* h: 2
* w: 3
* initial guess $(x,y) = (1.0, 1.0)$

In [ ]:
model_toldo = Model(Ipopt.Optimizer)
set_silent(model_toldo)

h_val = 2
w_val = 3

@variable(model_toldo, 0 <= x <= w_val, start = 1.0)
@variable(model_toldo, 0 <= y <= h_val, start = 1.0)

@NLobjective(model_toldo, Max, x * y)
@NLconstraint(model_toldo, (x/w_val)^2 + (y/h_val)^2 <= 1)

optimize!(model_toldo)

println("X ideal: ", round(value(x), digits=4))
println("Y ideal: ", round(value(y), digits=4))
println("Área Máxima de Sombra: ", round(objective_value(model_toldo), digits=4))

# Alocação de Designers

Problema de Maximização de competência.

In [ ]:
model_alocacao = Model(HiGHS.Optimizer)
set_silent(model_alocacao)

designers = 1:3
projetos = 1:4
scores = [90 80 10 50;
          60 70 50 65;
          70 40 80 85]
demandas = [70, 50, 85, 35]
h_max = 80

@variable(model_alocacao, h[designers, projetos] >= 0)
@objective(model_alocacao, Max, sum(h[i,j] * scores[i,j] for i in designers, j in projetos))

for i in designers
    @constraint(model_alocacao, sum(h[i,j] for j in projetos) <= h_max)
end
for j in projetos
    @constraint(model_alocacao, sum(h[i,j] for i in designers) >= demandas[j])
end

optimize!(model_alocacao)

println("Status: ", termination_status(model_alocacao))
println("Pontuação Total Máxima: ", objective_value(model_alocacao))
println("\nAlocação de Horas:")
for i in designers, j in projetos
    val = value(h[i,j])
    if val > 0
        println("Designer $i -> Projeto $j: $val horas")
    end
end

# Problema da Dieta

In [ ]:
model_dieta = Model(HiGHS.Optimizer)
set_silent(model_dieta)

@variable(model_dieta, y[1:4] >= 0)
@objective(model_dieta, Min, 1*y[1] + 0.5*y[2] + 2*y[3] + 0.3*y[4])

@constraint(model_dieta, 100*y[1] + 200*y[2] + 150*y[3] + 70*y[4] >= 500)
@constraint(model_dieta, 0.5*y[1] + 4*y[2] + 8*y[3] + 6*y[4] >= 50)
@constraint(model_dieta, 2*y[1] + 10*y[3] >= 100)

optimize!(model_dieta)

println("Custo Mínimo: ", round(objective_value(model_dieta), digits=2))
alimentos = ["Maçãs", "Pães", "Leite", "Ovos"]
for i in 1:4
    println("$(alimentos[i]): ", round(value(y[i]), digits=2))
end

# Problema da Mochila (Knapsack)

In [ ]:
model_mochila = Model(HiGHS.Optimizer)
set_silent(model_mochila)

@variable(model_mochila, x[1:3], Bin)
@objective(model_mochila, Max, 120*x[1] + 80*x[2] + 60*x[3])

@constraint(model_mochila, 2*x[1] + 1*x[2] + 1*x[3] <= 3.5)

optimize!(model_mochila)

println("Valor Total: ", objective_value(model_mochila))
itens = ["Barraca", "Fogareiro", "Comida"]
for i in 1:3
    if value(x[i]) > 0.5
        println("- Levar $(itens[i])")
    end
end